In [1]:
# Cell 0 — imports
import warnings
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42

In [2]:
BASE_DIR = Path(r"E:\Credit Risk Assessment System")
DATA_DIR = BASE_DIR / "dataset" / "processed_models"
CLEANED_DATA_PATH = BASE_DIR / "dataset" / "processed" / "cleaned_credit_data.csv"
MODEL_DIR = BASE_DIR / "models"
RESULT_DIR = BASE_DIR / "results"
PIPELINE_DIR = MODEL_DIR / "pipeline"

print("BASE_DIR:", BASE_DIR)

BASE_DIR: E:\Credit Risk Assessment System


In [3]:
# Cell 2 — load final model
MODEL_FILE_MAP = {
    "Logistic Regression": ("logistic_regression_model.pkl", "scaled"),
    "Decision Tree": ("decision_tree_model.pkl", "unscaled"),
    "Random Forest": ("random_forest_model.pkl", "unscaled"),
    "XGBoost": ("xgboost_model.pkl", "unscaled"),
    "Tuned XGBoost (GPU)": ("best_xgboost_model.pkl", "unscaled"),
}

final_summary = pd.read_csv(RESULT_DIR / "final_model_summary.csv")
final_model_name = final_summary["recommended_model"].iloc[0]
model_filename, data_type = MODEL_FILE_MAP[final_model_name]

model = joblib.load(MODEL_DIR / model_filename)

print("Final Model:", final_model_name)
print("Model File:", model_filename)
print("Model Loaded Successfully")

if type(model).__name__ == "XGBClassifier":
    device = model.get_xgb_params().get("device")
    if device == "cuda":
        print("GPU-enabled XGBoost model detected.")

Final Model: Tuned XGBoost (GPU)
Model File: best_xgboost_model.pkl
Model Loaded Successfully
GPU-enabled XGBoost model detected.


In [4]:
# Cell 3 — reconstruct encoder/scaler (fit on training data only, matches Notebook 5)
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cleaned_df = pd.read_csv(CLEANED_DATA_PATH)

subset_path = BASE_DIR / "dataset" / "processed" / "selected_top50.csv"
selected_features = (
    pd.read_csv(subset_path)["feature"].tolist() if subset_path.exists()
    else [c for c in cleaned_df.columns if c != "TARGET"]
)

X_full = cleaned_df[selected_features]
y_full = cleaned_df["TARGET"]

X_train_raw, _, y_train_raw, _ = train_test_split(
    X_full, y_full, test_size=0.2, random_state=RANDOM_STATE, stratify=y_full
)

categorical_features = X_train_raw.select_dtypes(include="object").columns.tolist()
numerical_features = X_train_raw.select_dtypes(include=np.number).columns.tolist()

encoder = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
        ("num", "passthrough", numerical_features),
    ]
)
encoder.fit(X_train_raw)

X_train_encoded = pd.DataFrame(
    encoder.transform(X_train_raw), columns=encoder.get_feature_names_out(), index=X_train_raw.index
)

scaler = StandardScaler()
scaler.fit(X_train_encoded)

print("Encoder/scaler reconstructed. Categorical:", len(categorical_features), "| Numerical:", len(numerical_features))

Encoder/scaler reconstructed. Categorical: 8 | Numerical: 42


In [5]:
# Cell 4 — expected feature schema (model's actual input columns)
EXPECTED_FEATURES = pd.read_csv(DATA_DIR / f"X_train_{data_type}.csv", nrows=0).columns.tolist()
print("Number of Expected Features:", len(EXPECTED_FEATURES))

Number of Expected Features: 149


In [6]:
# Cell 5 — raw field template (ratio ingredients + other raw fields the encoder needs)
REQUIRED_RAW_FIELDS = [
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
    "DAYS_BIRTH", "DAYS_EMPLOYED", "CNT_CHILDREN", "CNT_FAM_MEMBERS",
]
ENGINEERED_NAMES = ["AGE", "YEARS_EMPLOYED", "CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO",
                     "GOODS_CREDIT_RATIO", "EMPLOYMENT_AGE_RATIO", "INCOME_PER_CHILD",
                     "CREDIT_PER_CHILD", "ANNUITY_CREDIT_RATIO", "FAMILY_SIZE"]
OTHER_RAW_FIELDS = [c for c in selected_features if c not in ENGINEERED_NAMES and c not in REQUIRED_RAW_FIELDS]
RAW_FEATURE_TEMPLATE = REQUIRED_RAW_FIELDS + OTHER_RAW_FIELDS

RAW_NUMERIC_FIELDS = set(REQUIRED_RAW_FIELDS)
ALL_NUMERIC_FIELDS = set(numerical_features) | RAW_NUMERIC_FIELDS
ALL_CATEGORICAL_FIELDS = set(categorical_features) - RAW_NUMERIC_FIELDS

print("Raw feature template:", len(RAW_FEATURE_TEMPLATE), "fields")

Raw feature template: 45 fields


In [7]:
# Cell 6 — feature engineering (Notebook 3's exact formulas)
def safe_div(numerator, denominator):
    """Element-wise-safe division; returns NaN where denominator is 0."""
    return np.nan if denominator == 0 else numerator / denominator

def engineer_features(applicant: dict) -> dict:
    """Apply the exact feature engineering formulas from Notebook 3."""
    data = dict(applicant)
    data["AGE"] = round(-data["DAYS_BIRTH"] / 365, 1)
    years_employed = 0 if data["DAYS_EMPLOYED"] == 365243 else round(-data["DAYS_EMPLOYED"] / 365, 1)
    data["YEARS_EMPLOYED"] = years_employed
    data["CREDIT_INCOME_RATIO"] = safe_div(data["AMT_CREDIT"], data["AMT_INCOME_TOTAL"])
    data["ANNUITY_INCOME_RATIO"] = safe_div(data["AMT_ANNUITY"], data["AMT_INCOME_TOTAL"])
    data["GOODS_CREDIT_RATIO"] = safe_div(data["AMT_GOODS_PRICE"], data["AMT_CREDIT"])
    data["EMPLOYMENT_AGE_RATIO"] = safe_div(data["YEARS_EMPLOYED"], data["AGE"])
    data["INCOME_PER_CHILD"] = safe_div(data["AMT_INCOME_TOTAL"], data["CNT_CHILDREN"] + 1)
    data["CREDIT_PER_CHILD"] = safe_div(data["AMT_CREDIT"], data["CNT_CHILDREN"] + 1)
    data["ANNUITY_CREDIT_RATIO"] = safe_div(data["AMT_ANNUITY"], data["AMT_CREDIT"])
    data["FAMILY_SIZE"] = data["CNT_FAM_MEMBERS"]
    return data

In [8]:
# Cell 7 — demo applicant (raw fields pulled from X_train_raw, or cleaned_df if engineered-away)
def build_demo_applicant() -> dict:
    """A single realistic DEMONSTRATION APPLICANT — not a real customer."""
    template = {}
    for col in RAW_FEATURE_TEMPLATE:
        if col in X_train_raw.columns:
            template[col] = X_train_raw[col].median() if col in numerical_features else X_train_raw[col].mode().iloc[0]
        else:
            template[col] = cleaned_df[col].median() if pd.api.types.is_numeric_dtype(cleaned_df[col]) else cleaned_df[col].mode().iloc[0]
    template.update({
        "AMT_INCOME_TOTAL": 202500.0, "AMT_CREDIT": 406597.5, "AMT_ANNUITY": 24700.5,
        "AMT_GOODS_PRICE": 351000.0, "DAYS_BIRTH": -12005, "DAYS_EMPLOYED": -2000,
        "CNT_CHILDREN": 0, "CNT_FAM_MEMBERS": 2.0, "CODE_GENDER": "F",
    })
    return {k: v for k, v in template.items() if k in RAW_FEATURE_TEMPLATE}

demo_applicant = build_demo_applicant()
print("DEMONSTRATION APPLICANT (not a real customer):")
display(pd.Series(demo_applicant))

DEMONSTRATION APPLICANT (not a real customer):


AMT_INCOME_TOTAL                                    202500.0
AMT_CREDIT                                          406597.5
AMT_ANNUITY                                          24700.5
AMT_GOODS_PRICE                                     351000.0
DAYS_BIRTH                                            -12005
DAYS_EMPLOYED                                          -2000
CNT_CHILDREN                                               0
CNT_FAM_MEMBERS                                          2.0
EXT_SOURCE_1                                        0.505998
EXT_SOURCE_3                                        0.535276
EXT_SOURCE_2                                        0.565961
NAME_EDUCATION_TYPE            Secondary / secondary special
REGION_RATING_CLIENT_W_CITY                              2.0
DAYS_ID_PUBLISH                                      -3255.0
NAME_INCOME_TYPE                                     Working
REGION_RATING_CLIENT                                     2.0
NAME_HOUSING_TYPE       

In [9]:
# Cell 8 — validation
def validate_input(applicant: dict) -> list:
    """Return a list of validation error strings; empty list = valid."""
    errors = []
    for col in RAW_FEATURE_TEMPLATE:
        if col not in applicant:
            errors.append(f"Missing required field: '{col}'")
            continue
        value = applicant[col]
        if value is None or (isinstance(value, float) and np.isnan(value)):
            errors.append(f"Missing value for field: '{col}'")
            continue
        if col in ALL_NUMERIC_FIELDS and not isinstance(value, (int, float)):
            errors.append(f"'{col}' must be numeric, got {type(value).__name__}: {value!r}")
            continue
        if col in ALL_CATEGORICAL_FIELDS:
            valid_categories = set(X_train_raw[col].dropna().unique())
            if value not in valid_categories:
                errors.append(f"'{col}' has unknown category '{value}' (expected one of {sorted(valid_categories)[:5]}...)")
        if col in ("AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE", "CNT_CHILDREN", "CNT_FAM_MEMBERS"):
            if isinstance(value, (int, float)) and value < 0:
                errors.append(f"'{col}' cannot be negative, got {value}")
        if col in ("DAYS_BIRTH", "DAYS_EMPLOYED"):
            if isinstance(value, (int, float)) and value > 0 and value != 365243:
                errors.append(f"'{col}' should be negative (days before application), got {value}")

    extra_fields = set(applicant.keys()) - set(RAW_FEATURE_TEMPLATE)
    if extra_fields:
        errors.append(f"Unexpected fields not part of the schema: {sorted(extra_fields)}")
    return errors

print("Validation errors for demo applicant:", validate_input(demo_applicant) or "None — valid")

Validation errors for demo applicant: None — valid


In [10]:
# Cell 9 — preprocessing (validate -> engineer -> encode -> scale if needed -> align)
def preprocess_input(applicant: dict) -> pd.DataFrame:
    """Validate -> engineer -> encode -> scale (if needed) -> align columns."""
    errors = validate_input(applicant)
    if errors:
        raise ValueError("Input validation failed:\n" + "\n".join(errors))

    engineered = engineer_features(applicant)
    row = pd.DataFrame([engineered])[selected_features]

    encoded = pd.DataFrame(
        encoder.transform(row), columns=encoder.get_feature_names_out(), index=row.index
    )
    if data_type == "scaled":
        encoded = pd.DataFrame(scaler.transform(encoded), columns=encoded.columns, index=encoded.index)

    return encoded.reindex(columns=EXPECTED_FEATURES, fill_value=0)

model_ready_row = preprocess_input(demo_applicant)
print("Model-ready shape:", model_ready_row.shape)

Model-ready shape: (1, 149)


In [11]:
# Cell 10 — risk category + prediction function
def get_risk_category(probability: float, low_threshold: float = 0.30, high_threshold: float = 0.60) -> str:
    """DEMONSTRATION thresholds only — not bank-approved regulatory thresholds."""
    if probability < low_threshold:
        return "LOW RISK"
    if probability < high_threshold:
        return "MEDIUM RISK"
    return "HIGH RISK"

def predict_credit_risk(applicant: dict) -> dict:
    """Full pipeline: validate -> preprocess -> predict -> risk category."""
    X_row = preprocess_input(applicant)
    predicted_class = int(model.predict(X_row)[0])
    probability_default = float(model.predict_proba(X_row)[0, 1])
    risk_category = get_risk_category(probability_default)
    return {
        "predicted_class": "High Risk" if predicted_class == 1 else "Low Risk",
        "probability_of_default": probability_default,
        "risk_category": risk_category,
    }

result = predict_credit_risk(demo_applicant)
print("TARGET=0 means predicted non-default. TARGET=1 means predicted payment difficulty/default.")
result

TARGET=0 means predicted non-default. TARGET=1 means predicted payment difficulty/default.


{'predicted_class': 'Low Risk',
 'probability_of_default': 0.14637207984924316,
 'risk_category': 'LOW RISK'}

In [12]:
# Cell 11 — formatted single test
r = predict_credit_risk(demo_applicant)
print(f"""
Credit Risk Assessment

Predicted Class:
{r['predicted_class']}

Model-Predicted Probability of Default:
{r['probability_of_default']*100:.1f}%

Risk Category:
{r['risk_category']}
""")


Credit Risk Assessment

Predicted Class:
Low Risk

Model-Predicted Probability of Default:
14.6%

Risk Category:
LOW RISK



In [13]:
# Cell 12 — three demonstration applicants
demo_applicants = {
    "Lower-Risk Example": {**demo_applicant, "AMT_INCOME_TOTAL": 350000.0, "AMT_CREDIT": 150000.0,
                            "DAYS_EMPLOYED": -5000, "CODE_GENDER": "F"},
    "Medium-Risk Example": dict(demo_applicant),
    "Higher-Risk Example": {**demo_applicant, "AMT_INCOME_TOTAL": 90000.0, "AMT_CREDIT": 500000.0,
                             "DAYS_EMPLOYED": -200, "CNT_CHILDREN": 3},
}

test_rows = []
for label, applicant in demo_applicants.items():
    r = predict_credit_risk(applicant)
    test_rows.append({
        "Applicant": label, "Predicted Class": r["predicted_class"],
        "Probability of Default": round(r["probability_of_default"], 4),
        "Risk Category": r["risk_category"],
    })

test_results_df = pd.DataFrame(test_rows)
display(test_results_df)

,Applicant,Predicted Class,Probability of Default,Risk Category
0,Lower-Risk Example,Low Risk,0.0561,LOW RISK
1,Medium-Risk Example,Low Risk,0.1464,LOW RISK
2,Higher-Risk Example,Low Risk,0.2678,LOW RISK


In [14]:
# Cell 13 — SHAP explanation
import shap

_background = X_train_encoded.sample(min(100, len(X_train_encoded)), random_state=RANDOM_STATE)
if data_type == "scaled":
    _background = pd.DataFrame(scaler.transform(_background), columns=_background.columns)
_background = _background.reindex(columns=EXPECTED_FEATURES, fill_value=0)

_shap_explainer = shap.Explainer(model.predict_proba, _background)

def explain_prediction(applicant: dict, top_n: int = 5) -> dict:
    """Return the top positive/negative SHAP contributors for one applicant."""
    X_row = preprocess_input(applicant)
    shap_out = _shap_explainer(X_row)
    values = shap_out.values[0, :, 1]
    contrib = pd.Series(values, index=EXPECTED_FEATURES).sort_values()
    return {"top_increasing_risk": contrib.tail(top_n)[::-1], "top_decreasing_risk": contrib.head(top_n)}

explanation = explain_prediction(demo_applicants["Higher-Risk Example"])
print("Top factors increasing predicted risk:")
display(explanation["top_increasing_risk"])
print("Top factors decreasing predicted risk:")
display(explanation["top_decreasing_risk"])
print("\nNote: SHAP describes model behavior, not causality.")

PermutationExplainer explainer: 2it [00:13, 13.76s/it]               

Top factors increasing predicted risk:


cat__CODE_GENDER_F           0.048111
num__ANNUITY_CREDIT_RATIO    0.043105
num__GOODS_CREDIT_RATIO      0.036744
cat__FLAG_OWN_CAR_N          0.021040
num__AMT_GOODS_PRICE         0.017835
dtype: float64

Top factors decreasing predicted risk:


cat__CODE_GENDER_M            -0.059856
num__EXT_SOURCE_2             -0.026116
num__EXT_SOURCE_3             -0.020580
cat__FLAG_OWN_CAR_Y           -0.013385
num__REG_CITY_NOT_WORK_CITY   -0.002257
dtype: float64


Note: SHAP describes model behavior, not causality.


In [15]:
# Cell 14 — pipeline timing
def timed_predict(applicant: dict) -> dict:
    t0 = time.time()
    validate_input(applicant)
    t1 = time.time()
    X_row = preprocess_input(applicant)
    t2 = time.time()
    model.predict(X_row)
    t3 = time.time()
    return {
        "Input Validation Time (s)": round(t1 - t0, 5),
        "Preprocessing Time (s)": round(t2 - t1, 5),
        "Model Prediction Time (s)": round(t3 - t2, 5),
        "Total Pipeline Time (s)": round(t3 - t0, 5),
    }

display(pd.Series(timed_predict(demo_applicant)))

Input Validation Time (s)    0.21201
Preprocessing Time (s)       0.17800
Model Prediction Time (s)    0.02498
Total Pipeline Time (s)      0.41500
dtype: float64

In [16]:
# Cell 15 — error handling tests
invalid_cases = {
    "Missing required field": {k: v for k, v in demo_applicant.items() if k != "AMT_INCOME_TOTAL"},
    "Invalid numeric value": {**demo_applicant, "AMT_INCOME_TOTAL": "not_a_number"},
    "Unknown categorical value": {**demo_applicant, "CODE_GENDER": "UNKNOWN_VALUE"},
    "Negative amount": {**demo_applicant, "AMT_CREDIT": -1000},
}
for label, bad_input in invalid_cases.items():
    print(f"--- {label} ---")
    try:
        predict_credit_risk(bad_input)
        print("No error raised (unexpected)")
    except ValueError as exc:
        print(f"Caught error as expected:\n{exc}\n")

--- Missing required field ---
Caught error as expected:
Input validation failed:
Missing required field: 'AMT_INCOME_TOTAL'

--- Invalid numeric value ---
Caught error as expected:
Input validation failed:
'AMT_INCOME_TOTAL' must be numeric, got str: 'not_a_number'

--- Unknown categorical value ---
Caught error as expected:
Input validation failed:
'CODE_GENDER' has unknown category 'UNKNOWN_VALUE' (expected one of ['F', 'M', 'XNA']...)

--- Negative amount ---
Caught error as expected:
Input validation failed:
'AMT_CREDIT' cannot be negative, got -1000



In [17]:
# Cell 16 — save artifacts (incl. raw_feature_metadata + defaults, needed by app.py)
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(model, PIPELINE_DIR / "final_model.pkl")
joblib.dump({"encoder": encoder, "scaler": scaler, "data_type": data_type}, PIPELINE_DIR / "preprocessor.pkl")
joblib.dump(EXPECTED_FEATURES, PIPELINE_DIR / "feature_schema.pkl")

risk_thresholds = {"low_threshold": 0.30, "high_threshold": 0.60,
                    "note": "Demonstration thresholds only — not bank-approved regulatory thresholds."}
(PIPELINE_DIR / "risk_thresholds.json").write_text(json.dumps(risk_thresholds, indent=2))

raw_feature_defaults = {
    col: (X_train_raw[col].median() if col in numerical_features else X_train_raw[col].mode().iloc[0])
    for col in RAW_FEATURE_TEMPLATE if col in X_train_raw.columns
}
raw_feature_defaults.update({
    col: (cleaned_df[col].median() if pd.api.types.is_numeric_dtype(cleaned_df[col]) else cleaned_df[col].mode().iloc[0])
    for col in RAW_FEATURE_TEMPLATE if col not in raw_feature_defaults
})

raw_feature_metadata = {
    "raw_feature_template": RAW_FEATURE_TEMPLATE,
    "numerical_fields": sorted(ALL_NUMERIC_FIELDS),
    "categorical_fields": sorted(ALL_CATEGORICAL_FIELDS),
    "categorical_choices": {col: sorted(X_train_raw[col].dropna().unique().tolist())
                             for col in ALL_CATEGORICAL_FIELDS if col in X_train_raw.columns},
    "defaults": raw_feature_defaults,
    "final_model_name": final_model_name,
}
joblib.dump(raw_feature_metadata, PIPELINE_DIR / "raw_feature_metadata.pkl")

print("Saved all artifacts to", PIPELINE_DIR)

Saved all artifacts to E:\Credit Risk Assessment System\models\pipeline


In [18]:
# Cell 17 — reusable pipeline class
class CreditRiskPipeline:
    """Reusable credit-risk prediction pipeline for a future application."""

    def __init__(self, pipeline_dir: Path = PIPELINE_DIR):
        self.pipeline_dir = pipeline_dir
        self.model = self.encoder = self.scaler = self.data_type = None
        self.expected_features = self.thresholds = None

    def load_artifacts(self):
        self.model = joblib.load(self.pipeline_dir / "final_model.pkl")
        artifacts = joblib.load(self.pipeline_dir / "preprocessor.pkl")
        self.encoder, self.scaler, self.data_type = artifacts["encoder"], artifacts["scaler"], artifacts["data_type"]
        self.expected_features = joblib.load(self.pipeline_dir / "feature_schema.pkl")
        self.thresholds = json.loads((self.pipeline_dir / "risk_thresholds.json").read_text())
        return self

    def validate_input(self, applicant):
        return validate_input(applicant)

    def engineer_features(self, applicant):
        return engineer_features(applicant)

    def preprocess(self, applicant):
        return preprocess_input(applicant)

    def predict(self, applicant):
        return predict_credit_risk(applicant)

    def explain(self, applicant, top_n=5):
        return explain_prediction(applicant, top_n)

    def get_risk_category(self, probability):
        return get_risk_category(probability, self.thresholds["low_threshold"], self.thresholds["high_threshold"])

pipeline = CreditRiskPipeline().load_artifacts()
print("CreditRiskPipeline loaded and ready.")

CreditRiskPipeline loaded and ready.


In [19]:
# Cell 18 — end-to-end test
for label, applicant in demo_applicants.items():
    r = pipeline.predict(applicant)
    print(f"{label}: {r['predicted_class']} | {r['probability_of_default']*100:.1f}% | {r['risk_category']}")

Lower-Risk Example: Low Risk | 5.6% | LOW RISK
Medium-Risk Example: Low Risk | 14.6% | LOW RISK
Higher-Risk Example: Low Risk | 26.8% | LOW RISK


In [20]:
# Cell 19 — save test results
rows = []
for label, applicant in demo_applicants.items():
    start = time.time()
    r = pipeline.predict(applicant)
    elapsed = time.time() - start
    rows.append({"Applicant": label, "Predicted Class": r["predicted_class"],
                 "Probability": round(r["probability_of_default"], 4),
                 "Risk Category": r["risk_category"], "Prediction Time (s)": round(elapsed, 5)})

pd.DataFrame(rows).to_csv(RESULT_DIR / "prediction_pipeline_test_results.csv", index=False)
print("Saved prediction_pipeline_test_results.csv to", RESULT_DIR)

Saved prediction_pipeline_test_results.csv to E:\Credit Risk Assessment System\results
